# 08 — Antibiogram-conflict comparison (replicate a documented FedAvg failure condition)

Runs the **same protocol twice** on the pre-registered `antibiogram_conflict` split —
once with plain FedAvg, once with the topology-aware strategy — judged against the
criteria fixed BEFORE any run in `docs/2026-10-18-antibiogram-conflict-split.md`.
Read that doc first: it defines C1/C2/C3 and the binding falsification rules.

**Before running:**
1. Runtime → **GPU** (high-RAM helps; full 1.6M-row cohort).
2. Run the **smoke cell first** (minutes, no training). If an assert fails, STOP —
   the split lacks enough conflict on real data; fix the split BEFORE burning GPU
   hours. That is what the smoke gate is for.
3. Arms are 5 seeds x 2 arms (v2 protocol: 8 rounds x 4 epochs, hidden 128) —
   a few hours each. Each arm is one cell, so a failure loses nothing else.
4. Gates: **C1** FedAvg hurts the minority (worst_fed <= worst_local - 0.01) —
   the literature condition reproduced; **C2** Path-B gate (topo >= +0.005 on
   worst-hospital or mean); **C3** topology must RAISE the tail hospital.


In [ ]:
# 1) Get the code + deps
!git clone -b main https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout main && git pull)
!pip install -q torch_geometric 'flwr[simulation]'


In [ ]:
# 2) Point at the data (mount Drive, set ARMD_DIR before importing amr_fed)
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'   # EDIT to your ARMD folder


In [ ]:
# 3) Sanity: data resolves + imports
import sys
sys.path.insert(0, '/content/Capstone-amr-fed/src')
from amr_fed import config
from pathlib import Path
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong — fix cell 2 and re-run.'
from amr_fed.federated.run import run_fedavg, run_multiseed
from amr_fed.data_loader import load_cohort_frame, PK
from amr_fed.partition import antibiogram_conflict, partition_summary, _patient_shared_rate
print('imports OK')


In [ ]:
# 4) Load the cohort ONCE and reuse it for every arm
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.data_loader import load_cohort_frame
df = load_cohort_frame()
print('rows:', f'{len(df):,}')


## Smoke gate — partition validity only (no training, minutes)

Prints per-hospital sizes + antibiograms and asserts the pre-registered validity
checks: 1:1 patient assignment, every hospital >= 3% of tests, resistance-rate
spread >= 0.15. **If an assert fails, stop here and report the output.**


In [ ]:
# 5) SMOKE: partition validity + per-hospital antibiograms (no training)
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.data_loader import load_cohort_frame, PK
from amr_fed.partition import antibiogram_conflict, partition_summary, _patient_shared_rate

assign = antibiogram_conflict(df, n_clients=5)
g, niid = partition_summary(df, assign)
print(g, "\nnon-IID(wstd):", round(niid, 4))
assert assign.index.is_unique and len(assign) == df[PK].nunique(), 'not 1:1'
assert (g['n_tests'] / g['n_tests'].sum() >= 0.03).all(), 'hospital below 3% size floor'
spread = g['resist_rate'].max() - g['resist_rate'].min()
assert spread >= 0.15, f'rate spread {spread:.4f} < 0.15 — split lacks conflict'
print(f'rate spread OK: {spread:.4f}')
print('SMOKE PASS — safe to run the arms.')


## Arms (5 seeds each, v2 protocol: 8 rounds x 4 epochs = 32 budget, hidden 128)


In [ ]:
# 6) ANTIBIOGRAM-CONFLICT, FedAvg arm (5 seeds)
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.partition import antibiogram_conflict
ab_fed = run_multiseed(seeds=(42, 43, 44, 45, 46), df=df, label='antibiogram/FedAvg',
                       partition_fn=lambda d, s: antibiogram_conflict(d, n_clients=5, seed=s),
                       n_clients=5, rounds=8, local_epochs=4,
                       local_only_epochs=32, hidden=128)


In [ ]:
# 7) ANTIBIOGRAM-CONFLICT, topology arm (same protocol, distinctiveness T=1.0)
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.partition import antibiogram_conflict
ab_topo = run_multiseed(seeds=(42, 43, 44, 45, 46), df=df, label='antibiogram/TOPO',
                        partition_fn=lambda d, s: antibiogram_conflict(d, n_clients=5, seed=s),
                        n_clients=5, rounds=8, local_epochs=4,
                        local_only_epochs=32, hidden=128, strategy='topology',
                        topo_mode='distinctiveness', topo_temperature=1.0)


## Verdict — pre-registered C1/C2/C3

Run after BOTH arms. Prints the three criteria and the per-seed tail-hospital F1s,
saves JSON to Drive, and prints a COPY block — paste it back into chat.


In [ ]:
# 8) VERDICT: pre-registered C1/C2/C3 (docs/2026-10-18-antibiogram-conflict-split.md)
import json
import numpy as np

def _tail_f1(res, key):
    vals = []
    for r in res['runs']:
        pc = r.get(key)
        if not pc:
            continue
        vals.append(pc[max(pc, key=lambda x: int(x))] if isinstance(pc, dict) else pc[-1])
    return vals

d_mean = round(ab_topo['fedavg_best'][0] - ab_fed['fedavg_best'][0], 4)
d_worst = round(ab_topo['worst_fed'][0] - ab_fed['worst_fed'][0], 4)
c1 = ab_fed['worst_fed'][0] <= ab_fed['worst_local'][0] - 0.01
c2 = (d_worst >= 0.005) or (d_mean >= 0.005)

print(f"fedavg  best {ab_fed['fedavg_best']} | worst {ab_fed['worst_fed']} "
      f"| worst_local {ab_fed['worst_local']}")
print(f"topo    best {ab_topo['fedavg_best']} | worst {ab_topo['worst_fed']}")
print(f"C1 FedAvg hurts minority (worst_fed <= worst_local - 0.01): "
      f"{'PASS' if c1 else 'FAIL'}")
print(f"C2 Path-B gate (delta_mean {d_mean} | delta_worst {d_worst}): "
      f"{'PASS' if c2 else 'FAIL'}")

tail_f = _tail_f1(ab_fed, 'fedavg_per_client')
tail_t = _tail_f1(ab_topo, 'fedavg_per_client')
tail_l = _tail_f1(ab_fed, 'local_only_per_client')
tf = tt = tl = None
if tail_f and tail_t:
    tf, tt = round(float(np.mean(tail_f)), 4), round(float(np.mean(tail_t)), 4)
    tl = round(float(np.mean(tail_l)), 4) if tail_l else None
    print(f"C3 tail hospital F1: local {tl} | fedavg {tf} | topo {tt} -> "
          f"{'PASS' if tt > tf else 'FAIL'} (topo must RAISE the tail)")

payload = {'C1': bool(c1), 'C2': bool(c2), 'delta_mean': d_mean, 'delta_worst': d_worst,
           'tail_local': tl, 'tail_fedavg': tf, 'tail_topo': tt,
           'tail_fedavg_per_seed': tail_f, 'tail_topo_per_seed': tail_t,
           'fed': {'best': ab_fed['fedavg_best'], 'worst': ab_fed['worst_fed'],
                   'worst_local': ab_fed['worst_local'], 'final': ab_fed['fedavg_final']},
           'topo': {'best': ab_topo['fedavg_best'], 'worst': ab_topo['worst_fed'],
                    'final': ab_topo['fedavg_final']}}
path = '/content/drive/MyDrive/amr_antibiogram_verdict.json'
with open(path, 'w') as f:
    json.dump(payload, f, default=str, indent=1)
print('saved', path)
print('=== COPY EVERYTHING BELOW THIS LINE BACK TO CHAT ===')
print(json.dumps(payload, default=str))
print('=== END ===')
